# 02. Data Processing

Load raw CSV, clean, engineer features, split, scale, and persist arrays + scaler.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [ ]:
raw_path = os.path.join("..", "data", "iris_raw.csv")
df = pd.read_csv(raw_path)
print("Shape:", df.shape)
df.head()


In [ ]:
# Drop duplicate rows
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_dup} duplicate rows; new shape: {df.shape}")


In [ ]:
# IQR-based outlier flagging (per numeric feature); report and optionally remove
feature_cols = [c for c in df.columns if c not in ("target", "species")]
outlier_mask = np.zeros(len(df), dtype=bool)
for col in feature_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    col_out = (df[col] < low) | (df[col] > high)
    outlier_mask |= col_out
    print(f"{col}: {col_out.sum()} outliers (IQR)")

print(f"Total rows with any IQR outlier flag: {outlier_mask.sum()}")
# Remove rows flagged as outliers in any feature (common simple approach)
df_clean = df.loc[~outlier_mask].reset_index(drop=True)
print("Shape after removing outlier rows:", df_clean.shape)


In [ ]:
df = df_clean
# Feature engineering
df["petal_area"] = df["petal length (cm)"] * df["petal width (cm)"]
df["sepal_area"] = df["sepal length (cm)"] * df["sepal width (cm)"]
df["petal_sepal_ratio"] = df["petal_area"] / (df["sepal_area"] + 1e-8)
feature_cols = [c for c in df.columns if c not in ("target", "species")]
print("Features:", feature_cols)


In [ ]:
X = df[feature_cols].values
le = LabelEncoder()
y = le.fit_transform(df["species"])  # consistent with numeric target if species column exists

# 60% train / 20% val / 20% test (stratified)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


In [ ]:
proc_dir = os.path.join("..", "data", "processed")
os.makedirs(proc_dir, exist_ok=True)
models_dir = os.path.join("..", "models")
os.makedirs(models_dir, exist_ok=True)

np.save(os.path.join(proc_dir, "X_train.npy"), X_train_s)
np.save(os.path.join(proc_dir, "y_train.npy"), y_train)
np.save(os.path.join(proc_dir, "X_val.npy"), X_val_s)
np.save(os.path.join(proc_dir, "y_val.npy"), y_val)
np.save(os.path.join(proc_dir, "X_test.npy"), X_test_s)
np.save(os.path.join(proc_dir, "y_test.npy"), y_test)
# also save feature names for deployment
import json
with open(os.path.join(proc_dir, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)
with open(os.path.join(proc_dir, "label_classes.json"), "w", encoding="utf-8") as f:
    json.dump(list(le.classes_), f, indent=2)

scaler_path = os.path.join(models_dir, "scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print("Saved .npy arrays to", proc_dir)
print("Saved scaler to", scaler_path)
